Notebook to compare the first compilation (in icon-build0) with a compilation without the comin flag enabled (in icon-build). the first compilation had spikes in the 'LW net flux at TOA' plot, we expect a smooth plot....
The recompilation did NOT solve the issue...



## Necessary libraries

In [ ]:
%matplotlib inline

# system libs
import os, sys, glob
import datetime

# array operators and netcdf datasets
import numpy as np
import xarray as xr
xr.set_options(keep_attrs=True)

# plotting
import pylab as plt
import seaborn as sns
sns.set_context('talk')

import matplotlib.dates as mdates
myFmt = mdates.DateFormatter('%H:%M')

# to have tools to format time
sys.path.append( '/work/bb1224/2024_MS-COURSE/tools/analysis' )
from tools import convert_timevec

import warnings
warnings.simplefilter("ignore")


## Define plot function (added wanted time range)

In [ ]:
def plot_variable_mean(datasets, variable, title, mean_dims=('lat', 'lon'),time_range=None):
    """
    Plots the mean of a specified variable over given dimensions for multiple datasets.
    
    Parameters:
        datasets (dict): Dictionary where keys are dataset names and values are xarray datasets.
        variable (str): The name of the variable to plot.
        title (str): Title for the plot.
        mean_dims (tuple): Dimensions over which to compute the mean.
        time_range (tuple): Optional. A tuple of two datetime-like strings (start, end).(time_range=("2024-08-06T12:00", "2024-08-06T13:00"))
    """
    plt.figure(figsize=(10, 6))
    
    """for name, ds in datasets.items():
        if variable in ds:
            var_mean = ds[variable].mean(dim=mean_dims)
            var_mean.plot(label=name)
        else:
            print(f"Warning: Variable '{variable}' not found in dataset '{name}'")
    """
    for name, ds in datasets.items():
        if variable in ds:
            var = ds[variable]
            
            # Filter by time if specified
            if time_range is not None:
                start, end = time_range
                var = var.sel(time=slice(start, end))
            
            var_mean = var.mean(dim=mean_dims)
            var_mean.plot(label=name)
        else:
            print(f"Warning: Variable '{variable}' not found in dataset '{name}'")
    
    
    plt.title(title)
    plt.legend()
    plt.show()


## Comparison for original exp001-testruns:

### plot run from 12:00 to 13:00 from build0:

In [ ]:
# Path
sim_path = '/home/b/b383413/workspace/icon-build0/experiments/cesar1-20240806-exp001'


# ICON simulations for 3 nests:
# 2D variables
ds_2dicon1 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon2 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon3 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM03_ML_20240806T????00Z_regrid1km.nc')

# 3D variables
ds_3dicon1 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon2 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon3 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM03_ML_20240806T????00Z_regrid1km.nc')


# Define dictionaries to store datasets
datasets2d = {
    "exp001 icon d1": ds_2dicon1,
    "exp001 icon d2": ds_2dicon2,
    "exp001 icon d3": ds_2dicon3,
}

datasets3d = {
    "exp001 icon d1": ds_3dicon1,
    "exp001 icon d2": ds_3dicon2,
    "exp001 icon d3": ds_3dicon3,
}


# Define the spatial range over Lindenberg:
lat_min, lat_max = 51.85, 52.55   
lon_min, lon_max = 13.65, 14.55 

# Apply spatial selection to all datasets
for datasets in [datasets2d, datasets3d]:
    for name, ds in datasets.items():
        datasets[name] = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))


# Format time 
for name, ds in datasets2d.items():
    ds['time'] = convert_timevec(ds.time.data)
for name, ds in datasets3d.items():
    ds['time'] = convert_timevec(ds.time.data)

plot_variable_mean(datasets2d, 'thb_t', "LW net flux at TOA for Build0", mean_dims=('lat', 'lon'))

### plot run from 12:00 to 13:00 from new compilation build:

In [ ]:
sim_path = '/home/b/b383413/workspace/icon-build/experiments/cesar1-20240806-exp001'

# ICON simulations for 3 nests:
# 2D variables
ds_2dicon1 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon2 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon3 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM03_ML_20240806T????00Z_regrid1km.nc')

# 3D variables
ds_3dicon1 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon2 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon3 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM03_ML_20240806T????00Z_regrid1km.nc')


# Define dictionaries to store datasets
datasets2d = {
    "exp001 icon d1": ds_2dicon1,
    "exp001 icon d2": ds_2dicon2,
    "exp001 icon d3": ds_2dicon3,
}

datasets3d = {
    "exp001 icon d1": ds_3dicon1,
    "exp001 icon d2": ds_3dicon2,
    "exp001 icon d3": ds_3dicon3,
}


# Define the spatial range over Lindenberg:
lat_min, lat_max = 51.85, 52.55   
lon_min, lon_max = 13.65, 14.55 

# Apply spatial selection to all datasets
for datasets in [datasets2d, datasets3d]:
    for name, ds in datasets.items():
        datasets[name] = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))


# Format time 
for name, ds in datasets2d.items():
    ds['time'] = convert_timevec(ds.time.data)
for name, ds in datasets3d.items():
    ds['time'] = convert_timevec(ds.time.data)

plot_variable_mean(datasets2d, 'thb_t', "LW net flux at TOA for new build/compilation", mean_dims=('lat', 'lon'))

## Comparison from extended runs:

### plot run from 12:00 to 15:00 from build0 (with selecting desired timeframe):

In [ ]:
sim_path = '/home/b/b383413/workspace/icon-build0/experiments/cesar1-20240806-exp001-T06-21'


# ICON simulations for 3 nests:
# 2D variables
ds_2dicon1 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon2 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon3 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM03_ML_20240806T????00Z_regrid1km.nc')

# 3D variables
ds_3dicon1 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon2 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon3 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM03_ML_20240806T????00Z_regrid1km.nc')


# Define dictionaries to store datasets
datasets2d = {
    "exp001 icon d1": ds_2dicon1,
    "exp001 icon d2": ds_2dicon2,
    "exp001 icon d3": ds_2dicon3,
}

datasets3d = {
    "exp001 icon d1": ds_3dicon1,
    "exp001 icon d2": ds_3dicon2,
    "exp001 icon d3": ds_3dicon3,
}


# Define the spatial range over Lindenberg:
lat_min, lat_max = 51.85, 52.55   
lon_min, lon_max = 13.65, 14.55 

# Apply spatial selection to all datasets
for datasets in [datasets2d, datasets3d]:
    for name, ds in datasets.items():
        datasets[name] = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))


# Format time 
for name, ds in datasets2d.items():
    ds['time'] = convert_timevec(ds.time.data)
for name, ds in datasets3d.items():
    ds['time'] = convert_timevec(ds.time.data)


#plot_variable_mean(datasets2d,'thb_t',"LW net flux at TOA (12:00–15:00)(data from long run with old compilation)", mean_dims=('lat', 'lon'))
plot_variable_mean(datasets2d,'thb_t',"LW net flux at TOA (12:00–15:00)(data from long run with old compilation)", mean_dims=('lat', 'lon'),time_range=("2024-08-06T12:00", "2024-08-06T15:00"))

### plot run from 12:00 to 15:00 from new compilation build:

In [ ]:
sim_path = '/home/b/b383413/workspace/icon-build/experiments/cesar1-20240806-exp001-T12-15'

# ICON simulations for 3 nests:
# 2D variables
ds_2dicon1 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon2 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_2dicon3 = xr.open_mfdataset(f'{sim_path}/2d_*_DOM03_ML_20240806T????00Z_regrid1km.nc')

# 3D variables
ds_3dicon1 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM01_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon2 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM02_ML_20240806T????00Z_regrid1km.nc')
ds_3dicon3 = xr.open_mfdataset(f'{sim_path}/3d_full_base_DOM03_ML_20240806T????00Z_regrid1km.nc')

# Define dictionaries to store datasets
datasets2d = {
    "exp001 icon d1": ds_2dicon1,
    "exp001 icon d2": ds_2dicon2,
    "exp001 icon d3": ds_2dicon3,
}

datasets3d = {
    "exp001 icon d1": ds_3dicon1,
    "exp001 icon d2": ds_3dicon2,
    "exp001 icon d3": ds_3dicon3,
}

# Define the spatial range over Lindenberg:
lat_min, lat_max = 51.85, 52.55   
lon_min, lon_max = 13.65, 14.55 

# Apply spatial selection to all datasets
for datasets in [datasets2d, datasets3d]:
    for name, ds in datasets.items():
        datasets[name] = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

# Format time 
for name, ds in datasets2d.items():
    ds['time'] = convert_timevec(ds.time.data)
for name, ds in datasets3d.items():
    ds['time'] = convert_timevec(ds.time.data)

plot_variable_mean(datasets2d, 'thb_t', "LW net flux at TOA (new run after recompilation without comin flag) ", mean_dims=('lat', 'lon'))